# Example 8.9.2 / 1: HiGHS LP Algorithm Options

Companion notebook for **8.9.2 LP and MIP Algorithm Options** in the AMPL lab Documentation, section "HiGHS LP Algorithm Options". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## HiGHS LP Algorithm Options

`alg:method` (aliases include `method`, `lp:method` and `solver`) is a finite family. This
example runs each value that works on a CPU on the steel model of
How LP Solvers Work (AMPL lab documentation, section 8.9). `pre:solve=off` turns off HiGHS's own presolve, which would
otherwise solve this tiny LP before any algorithm runs. In `data.py` and `run.py`, `ampl` is an
`AMPL` object that has already read `model.mod`. Setting
`send_statuses` to `0` keeps one run's basis from warm-starting the next, so each run starts
cold. `solver_msg` `0` suppresses the solver message, though the driver still echoes its
options, and the program prints one summary line per method from `solve_message`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["send_statuses"] = 0
ampl.option["solver_msg"] = 0
for method in ["choose", "simplex", "ipm", "ipx", "hipo", "pdlp", "hipdlp", "qpasm"]:
    ampl.solve(solver="highs", highs_options=f"alg:method={method} pre:solve=off")
    work = ", ".join(ampl.get_value("solve_message").splitlines()[1:])
    objective = ampl.obj["Total_Profit"].value()
    coils = ampl.get_value("Make['coils'].sstatus")
    print(f"{method}: {ampl.solve_result}, objective {objective:.4f}, coils {coils}; {work}")

HiGHS 1.14.0:   lp:method = choose
  pre:solve = off
choose: solved, objective 196400.0000, coils low; 1 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
simplex: solved, objective 196400.0000, coils low; 1 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = ipm
  pre:solve = off
ipm: solved, objective 196400.0000, coils low; 0 simplex iterations, 6 barrier iterations
HiGHS 1.14.0:   lp:method = ipx
  pre:solve = off
ipx: solved, objective 196400.0000, coils low; 0 simplex iterations, 5 barrier iterations
HiGHS 1.14.0:   lp:method = hipo
  pre:solve = off
hipo: solved, objective 196400.0000, coils low; 0 simplex iterations, 6 barrier iterations
HiGHS 1.14.0:   lp:method = pdlp
  pre:solve = off
pdlp: solved, objective 196400.0000, coils sup; 80 PDLP iterations
HiGHS 1.14.0:   lp:method = hipdlp
  pre:solve = off
  Warning code 1 while solving with HiGHS
hipdlp: solved?, objective 0.0000, coils sup; 0 simplex iterations, 0 b

### Pure AMPL: data.dat and model.run

The same loop in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option send_statuses 0;
option solver_msg 0;
for {m in {"choose", "simplex", "ipm", "ipx", "hipo", "pdlp", "hipdlp", "qpasm"}} {
    option highs_options ("alg:method=" & m & " pre:solve=off");
    solve;
    printf "%s: %s, objective %.4f, coils %s\n", m, solve_result, Total_Profit,
        Make['coils'].sstatus;
}

HiGHS 1.14.0:   lp:method = choose
  pre:solve = off
choose: solved, objective 196400.0000, coils low
HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
simplex: solved, objective 196400.0000, coils low
HiGHS 1.14.0:   lp:method = ipm
  pre:solve = off
ipm: solved, objective 196400.0000, coils low
HiGHS 1.14.0:   lp:method = ipx
  pre:solve = off
ipx: solved, objective 196400.0000, coils low
HiGHS 1.14.0:   lp:method = hipo
  pre:solve = off
hipo: solved, objective 196400.0000, coils low
HiGHS 1.14.0:   lp:method = pdlp
  pre:solve = off
pdlp: solved, objective 196400.0000, coils sup
HiGHS 1.14.0:   lp:method = hipdlp
  pre:solve = off
  Warning code 1 while solving with HiGHS
hipdlp: solved?, objective 0.0000, coils sup
HiGHS 1.14.0:   lp:method = qpasm
  pre:solve = off
qpasm: solved, objective 196400.0000, coils low


### Another way in amplpy

| `alg:method` value | Documented meaning | Result on this LP |
|---|---|---|
| `choose` (default) | Automatic | Chose simplex: 1 simplex iteration, basis returned (`low`) |
| `simplex` | Simplex | 1 simplex iteration, basis returned |
| `ipm` | Interior Point Method (automatic) | 6 barrier iterations, crossover gave a basis |
| `ipx` | IPX interior point method | 5 barrier iterations, crossover gave a basis |
| `hipo` | HiGHS interior point method | 6 barrier iterations, crossover gave a basis |
| `pdlp` | cuPDLP-c solver | 80 PDLP iterations, `solved`, no basis (`sup`) |
| `hipdlp` | HiPDLP solver | `solved?` with objective 0: a candidate, not an optimum. Always check `solve_result` |
| `qpasm` | QP active set method | On this LP the run reports 1 simplex iteration and a basis |
| `pdlp-gpu` | cuPDLP-c on an NVIDIA GPU; requires CUDA, not available on macOS | Not run for this page (no GPU) |
| `hipdlp-gpu` | HiPDLP on an NVIDIA GPU; requires CUDA, not available on macOS | Not run for this page (no GPU) |

Three more HiGHS options decide which simplex runs, whether crossover runs, and whether a basis
is exchanged at all:

| Option and value | Documented meaning (installed HiGHS 1.14.0 listing) |
|---|---|
| `alg:simplex=0` | Choose automatically (default) |
| `alg:simplex=1` | Dual simplex, serial |
| `alg:simplex=2` | Dual simplex, PAMI (parallelization across multiple iterations) |
| `alg:simplex=3` | Dual simplex, SIP (single iteration parallelism) |
| `alg:simplex=4` | Primal simplex |
| `bar:crossover=on` | Run crossover after IPM to get a basic solution |
| `bar:crossover=off` | Do not run crossover |
| `bar:crossover=choose` | Run crossover only if the IPM result without crossover is imprecise |
| `alg:basis=0` to `3` | Whether to use an incoming basis (`1`), return the final basis (`2`), both (`3`, default), or neither (`0`) |

With the same `model.mod` and `data.py`, this sweep runs those values:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.option["send_statuses"] = 0
ampl.option["solver_msg"] = 0
runs = [f"alg:method=simplex alg:simplex={k}" for k in range(5)] + [
    "alg:method=ipm bar:crossover=on",
    "alg:method=ipm bar:crossover=off",
    "alg:method=ipm bar:crossover=choose",
    "alg:method=pdlp bar:crossover=on",
    "alg:method=simplex alg:basis=0",
]
for opts in runs:
    ampl.solve(solver="highs", highs_options=opts + " pre:solve=off")
    work = ", ".join(ampl.get_value("solve_message").splitlines()[1:])
    coils = ampl.get_value("Make['coils'].sstatus")
    print(f"{opts}: {ampl.solve_result}, coils {coils}; {work}")

HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 0
  pre:solve = off
alg:method=simplex alg:simplex=0: solved, coils low; 2 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 1
  pre:solve = off
alg:method=simplex alg:simplex=1: solved, coils low; 1 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 2
  pre:solve = off
alg:method=simplex alg:simplex=2: solved, coils low; 1 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 3
  pre:solve = off
alg:method=simplex alg:simplex=3: solved, coils low; 1 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 4
  pre:solve = off
alg:method=simplex alg:simplex=4: solved, coils low; 2 simplex iterations, 0 barrier iterations
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = on
  pre:solve = off
alg:method=ipm bar:crossover=on: solved, coils low; 0 simplex iterations, 6 barrier itera

### Pure AMPL: data.dat and model.run

The same sweep in pure AMPL, with the same `model.mod` and `data.dat`:

In [12]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option send_statuses 0;
option solver_msg 0;
# HiGHS option strings to compare, one solve each
set RUNS ordered = {"alg:method=simplex alg:simplex=0", "alg:method=simplex alg:simplex=1",
    "alg:method=simplex alg:simplex=2", "alg:method=simplex alg:simplex=3",
    "alg:method=simplex alg:simplex=4", "alg:method=ipm bar:crossover=on",
    "alg:method=ipm bar:crossover=off", "alg:method=ipm bar:crossover=choose",
    "alg:method=pdlp bar:crossover=on", "alg:method=simplex alg:basis=0"};
for {opts in RUNS} {
    option highs_options (opts & " pre:solve=off");
    solve;
    printf "%s: %s, coils %s\n", opts, solve_result, Make['coils'].sstatus;
}

HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 0
  pre:solve = off
alg:method=simplex alg:simplex=0: solved, coils low
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 1
  pre:solve = off
alg:method=simplex alg:simplex=1: solved, coils low
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 2
  pre:solve = off
alg:method=simplex alg:simplex=2: solved, coils low
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 3
  pre:solve = off
alg:method=simplex alg:simplex=3: solved, coils low
HiGHS 1.14.0:   lp:method = simplex
  alg:simplex = 4
  pre:solve = off
alg:method=simplex alg:simplex=4: solved, coils low
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = on
  pre:solve = off
alg:method=ipm bar:crossover=on: solved, coils low
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = off
  pre:solve = off
alg:method=ipm bar:crossover=off: solved, coils sup
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = choose
  pre:solve = off
alg:method=ipm bar:crossover=choose: solved, coils sup
HiGH